In [ ]:
!pip install -q pandas
!pip install langchain
!pip install langchain-community
!pip install langchain-core
!pip install langchain-huggingface
!pip install langchain-google-genai
!pip install faiss-cpu
!pip install sentence-transformers

In [ ]:
import os
import pandas as pd
from getpass import getpass

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API Key: ")


/tmp/ipykernel_24337/2060359392.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Enter Gemini API Key: ··········


In [ ]:
df = pd.read_csv("/content/qa_data (1) (1).csv")
df = df.fillna("")

print("CSV Loaded Successfully!")
print("Total Records:", len(df))

CSV Loaded Successfully!
Total Records: 7


In [ ]:
documents=[]
for _,row in df.iterrows():
  text=f"""
  Question:{row['question']}
  Answer:{row['answer']}
  """
  documents.append(Document(page_content=text))
  print("Documents Created:", len(documents))

Documents Created: 1
Documents Created: 2
Documents Created: 3
Documents Created: 4
Documents Created: 5
Documents Created: 6
Documents Created: 7


In [ ]:
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore=FAISS.from_documents(
    documents,
    embeddings
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

print("FAISS Vector Database Created!")

FAISS Vector Database Created!


In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0
)

print("Gemini Model Connected!")

Gemini Model Connected!


In [ ]:
def ask_gemini(query):

    # Retrieve relevant documents
    results = vectorstore.similarity_search(
        query,
        k=3
    )

    # Extract retrieved context
    context = "\n\n".join(
        doc.page_content for doc in results
    )

    # Create prompt
    prompt = f"""
    You are a Q&A assistant.

    Answer ONLY using the retrieved context below.

    If the answer is not available in the context,
    say: No relevant Q&A found.

    Retrieved Context:
    {context}

    User Question:
    {query}

    Answer:
    """

    # Generate answer using Gemini
    response = llm.invoke(prompt)

    return response.content

In [ ]:
print("\nRAG Chatbot is Running!")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower().strip() == "exit":
        print("Goodbye!")
        break

    answer = ask_gemini(user_input)

    print("\nBot:", answer)
    print()


RAG Chatbot is Running!
Type 'exit' to stop.

You: what is your name?

Bot: [{'type': 'text', 'text': 'I am your custom chatbot.', 'extras': {'signature': 'EpEICo4IAWkUfRPvG2GRU4Aj1yI9+oVFFI6iX+SFniHkoIDdUEsSZPDZSLqDb+o2v40FVOHr8Xavl/tTXcxu3JRdGlMlFPxsvOtjkwiu/ivk1hZ2ptIGSw4ozKK1c6dQM1NF1JoIW61+IvuDZbCugUXm7aLQ/fq1BUMIozLRcMiJrF0UmZn8Lv2xvylfkjAmp/YJHNjxd4Yq2+B8S61prVPxhTUjACuFSqqk/1XL24TwwgPgkgC8Ef2jpZ3OVcInQDEYkgzqEiDUHLJ3baq+aOVSGvZPFG4qxAxGxWvSgeYWsxNmMeC/xu0LPhrDmTbqTRgu3uz+m3WvKHNltHcUC0ObMA6QBHYqp0hNLGNEm1CmF9ipgzPjHCL/aZXIXFV58rzmA+QiDfJHLTp7m2jBOrkoyPxtILdLbKxlGgxR5xo0dKn9C4rBxy34MRbhF0ezNiJw6dUKJJHwaAJ6ORSoHlTIARZqXl2szF62hF6UIxm9IkWQO14ppr6RYUdYd4tWBHYSnvDXmOaGt3FGJk0EM7hb+VmynaWyaF8DsHb3o8D99S0s8M2H/ti7znzcDGQhM9k/VVaZAn2DF9mlmbwPjjdglt86cNR1bP4ePIQqahvX0T1hcIFp2jftjMd/8rfoBPIrxQdDOhoqynuWhKwyI3/I+3KVVbTbdiPKYGPOSl4nrLAhYIdfcok5HNVhs1juomsKT+pIQNHbQjqPzoNC/r/C17kcf3O+D1VrFYS7SnwmcZjwRbK2NFMicnxjNENkBMo/pLK1tWd69f9E/m+sR4SWXn/Dwzi8HK5l2cch5CtTVbt53WwNYwReFkh5xhyMQMNH4wAzwDi